In [ ]:
import pandas as pd
import numpy as np

# Files
crash_path = "/Users/yiyuanemmazhou/Documents/New Project/bostoncrashsproad20182026.csv"
coef_path  = "/Users/yiyuanemmazhou/Documents/New Project/negbinomialregintersections.csv"

# Load
crash = pd.read_csv(crash_path)
coef  = pd.read_csv(coef_path)

# Columns
intersection_col = [c for c in crash.columns if "intersection" in c.lower()][0]
rs_col = [c for c in crash.columns if "road_surface" in c.lower()][0]

# Clean
crash = crash[crash[intersection_col].notna() & (crash[intersection_col].astype(str).str.strip() != "")]
crash[intersection_col] = crash[intersection_col].astype(str).str.strip()
crash[rs_col] = crash[rs_col].astype(str).str.strip().str.lower()

# Get coefficients -> IRR
coef_map = dict(zip(coef["Unnamed: 0"], coef["Coef."]))
beta_ice   = coef_map.get("C(road_surface)[T.Ice]", 0.0)
beta_slush = coef_map.get("C(road_surface)[T.Slush]", 0.0)

irr_ice   = float(np.exp(beta_ice))
irr_slush = float(np.exp(beta_slush))

# Counts
counts = crash.groupby(intersection_col).size().rename("N_i")
ice_counts   = crash[crash[rs_col] == "ice"].groupby(intersection_col).size().rename("ice_n")
slush_counts = crash[crash[rs_col] == "slush"].groupby(intersection_col).size().rename("slush_n")

# Merge + shares
df = pd.concat([counts, ice_counts, slush_counts], axis=1).fillna(0)
df["ice_share"]   = df["ice_n"] / df["N_i"]
df["slush_share"] = df["slush_n"] / df["N_i"]

# Priority metric
df["priority"] = df["N_i"] * (irr_ice * df["ice_share"] + irr_slush * df["slush_share"])

# Top 10% by quantile rule
M = df.shape[0]
q = 0.90
N = int(np.ceil((1 - q) * M))

df = df.reset_index().rename(columns={intersection_col: "intersection"})
df = df.sort_values("priority", ascending=False)

top10pct = df.head(N)

out = "/Users/yiyuanemmazhou/Documents/New Project/top10pct_priority_ice_slush.csv"
top10pct.to_csv(out, index=False)

print("Saved:", out)
print("M:", M, "N:", N)

In [ ]:
import pandas as pd

path = "/Users/yiyuanemmazhou/Documents/New Project/priority_ice_slush_with_volume_weight.csv"

df = pd.read_csv(path)

# Rank by weighted priority
df = df.sort_values("priority_ice_slush_weighted", ascending=False)
df["rank"] = range(1, len(df) + 1)

out = "/Users/yiyuanemmazhou/Documents/New Project/priority_ice_slush_with_volume_weight_ranked.csv"
df.to_csv(out, index=False)

print("Saved:", out)